# MTamba 2D Feasibility Study

**Paper:** *Tetra-Orientated Mamba with T2-FLAIR Mismatch Features for Glioma Segmentation, IDH Genotyping and Grading* — MICCAI 2025

## What we borrowed from the paper
- **Multi-task learning:** joint segmentation + IDH classification
- **Modality dropout:** randomly zero input channels during training to simulate missing sequences
- **Missing-modality evaluation:** test on FLAIR-only, T2+FLAIR, no-T1CE vs full-4
- **Uncertainty-aware triage:** MC-dropout confidence scores to decide auto-accept vs refer-to-radiologist

## What we simplified (2D proxy)
- **2D U-Net** replaces 3D Mamba (no custom CUDA kernel needed)
- **128 x 128 crops** instead of full 3D volumes (fits RTX 5060 Ti 16 GB)
- **No T2-FLAIR mismatch module** — kept for future 3D reproduction
- **Grade head removed** — our 50-subject subset was 100 % HGG

**Dataset:** UCSF-PDGM (n = 50 subjects, 4 MRI sequences + segmentation + IDH labels)

In [ ]:
import os, glob, json, math, random, time
from pathlib import Path
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score

CONFIG = {
    'data_dir': 'data',
    'work_dir': 'work',
    'channel_order': ['flair', 't1ce', 't1', 't2'],
    'label_map': {1: 1, 2: 2, 3: 4},
    'crop_size': 160,
    'out_size': 128,
    'min_tumor_vox': 50,
    'min_slices': 5,
    'drop_p': 0.3,
    'batch_size': 16,
    'epochs': 15,
    'lr': 3e-4,
    'weight_decay': 1e-5,
    'val_frac': 0.2,
    'seed': 42,
    'mc_passes': 20,
}

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])

os.makedirs(CONFIG['work_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['work_dir'], 'slices'), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Label loading

Clinical CSV from TCIA provides IDH status and WHO grade per subject.
Subject IDs are normalized from folder names (e.g. `UCSF-PDGM-0004_nifti` -> `4`).

In [ ]:
def norm_sid(s):
    m = re.search(r'(\d+)', str(s))
    return int(m.group(1)) if m else None

def load_labels(data_dir):
    csvs = glob.glob(os.path.join(data_dir, '*.csv'))
    if not csvs:
        raise FileNotFoundError('clinical CSV not found in data_dir')
    clin = pd.read_csv(csvs[0])
    id_col = next((c for c in clin.columns if 'pdgm' in c.lower()), clin.columns[0])
    idh_col = next(c for c in clin.columns if 'idh' in c.lower())
    grade_col = next(c for c in clin.columns if 'grade' in c.lower())
    lab = {}
    for _, r in clin.iterrows():
        sid = norm_sid(r[id_col])
        idh = 1 if 'mut' in str(r[idh_col]).lower() else 0
        g = str(r[grade_col])
        grade = 1 if ('4' in g or 'g4' in g.lower() or 'hgg' in g.lower() or 'high' in g.lower()) else 0
        lab[sid] = (idh, grade)
    return lab

labels = load_labels(CONFIG['data_dir'])
subjects = sorted([d for d in glob.glob(os.path.join(CONFIG['data_dir'], '*')) if os.path.isdir(d)])
len(subjects), len(labels)

## Preprocessing

For each subject:
1. Resolve 4 MRI sequences + tumor segmentation by filename keywords
2. Z-score normalize within brain mask
3. Extract 160 x 160 axial slices centered on tumor
4. Save compressed NPZ (image, label, idh, grade)

In [ ]:
import nibabel as nib

def find_nifti(subject_dir, keys, exclude=()):
    files = [f for f in glob.glob(os.path.join(subject_dir, '**', '*'), recursive=True)
             if f.lower().endswith(('.nii', '.nii.gz'))]
    for f in sorted(files):
        b = os.path.basename(f).lower()
        if any(k in b for k in keys) and not any(e in b for e in exclude):
            return f
    return None

def resolve_subject(subject_dir):
    return {
        'flair': find_nifti(subject_dir, ['flair']),
        't1ce': find_nifti(subject_dir, ['t1c', 't1ce', 't1gd', 't1_gd', 't1+c']),
        't1': find_nifti(subject_dir, ['t1'], exclude=['t1c', 't1ce', 't1gd', 't1_gd', 't1+c']),
        't2': find_nifti(subject_dir, ['t2'], exclude=['flair']),
        'seg': find_nifti(subject_dir, ['tumor_segmentation', 'tumor'], exclude=['parenchyma']),
    }

def zscore(vol):
    brain = vol[vol > 0]
    if brain.size < 100:
        return vol.astype(np.float32)
    out = (vol - brain.mean()) / (brain.std() + 1e-8)
    out[vol == 0] = 0
    return out.astype(np.float32)

def preprocess_subject(subject_dir, label_map):
    sid = norm_sid(os.path.basename(subject_dir.rstrip('/')))
    files = resolve_subject(subject_dir)
    if any(files[k] is None for k in CONFIG['channel_order'] + ['seg']):
        return sid, None
    vols = [zscore(np.asanyarray(nib.load(files[k]).dataobj)) for k in CONFIG['channel_order']]
    seg = np.asanyarray(nib.load(files['seg']).dataobj).astype(np.int16)
    shape = vols[0].shape
    if any(v.shape != shape for v in vols) or seg.shape != shape:
        return sid, None
    for k, v in label_map.items():
        seg[seg == k] = v
    C = CONFIG['crop_size']
    imgs, labs = [], []
    for z in range(shape[2]):
        lab_sl = seg[:, :, z]
        if (lab_sl > 0).sum() < CONFIG['min_tumor_vox']:
            continue
        ys, xs = np.where(lab_sl > 0)
        cy, cx = int(ys.mean()), int(xs.mean())
        y0, x0 = cy - C // 2, cx - C // 2
        y0, x0 = max(0, min(y0, shape[0] - C)), max(0, min(x0, shape[1] - C))
        sl = np.stack([v[y0:y0 + C, x0:x0 + C, z] for v in vols], axis=-1)
        imgs.append(sl.astype(np.float32))
        labs.append(lab_sl[y0:y0 + C, x0:x0 + C].astype(np.uint8))
    if len(imgs) < CONFIG['min_slices'] or sid not in labels:
        return sid, None
    idh, grade = labels[sid]
    out = os.path.join(CONFIG['work_dir'], 'slices', f'{sid}.npz')
    np.savez_compressed(out, image=np.stack(imgs), label=np.stack(labs), idh=idh, grade=grade)
    return sid, out

t0 = time.time()
ok, fail = [], []
for d in subjects:
    sid, out = preprocess_subject(d, CONFIG['label_map'])
    (ok if out else fail).append(sid)
print(f'done in {time.time()-t0:.0f}s | usable: {len(ok)} | skipped: {len(fail)}')
print('first failures:', fail[:10])

## Dataset

`SliceDataset` loads per-slice NPZ files.

**Modality dropout** (from MTamba paper): during training, each channel is independently dropped with probability `drop_p = 0.3`. If all channels are dropped, one is randomly restored. This forces the model to learn robust representations that do not depend on any single MRI sequence.

Standard augmentations: random horizontal flip, intensity scaling.

In [ ]:
class SliceDataset(torch.utils.data.Dataset):
    def __init__(self, files, train=True):
        self.files = files
        self.train = train
        self.index = []
        for f in files:
            with np.load(f) as z:
                self.index += [(f, i) for i in range(z['image'].shape[0])]

    def __len__(self):
        return len(self.index)

    def crop(self, img, lab):
        C, O = CONFIG['crop_size'], CONFIG['out_size']
        m = (C - O) // 2
        if self.train:
            dy, dx = np.random.randint(0, 2 * m + 1, size=2)
        else:
            dy, dx = m, m
        return img[dy:dy + O, dx:dx + O], lab[dy:dy + O, dx:dx + O]

    def __getitem__(self, k):
        f, i = self.index[k]
        with np.load(f) as z:
            img, lab, idh = z['image'][i], z['label'][i], int(z['idh'])
        img, lab = self.crop(img, lab)
        if self.train:
            if np.random.rand() < 0.5:
                img = img[:, ::-1].copy()
                lab = lab[:, ::-1].copy()
            img = img * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.1, 0.1)
            keep = np.random.rand(4) >= CONFIG['drop_p']
            if not keep.any():
                keep[np.random.randint(4)] = True
        else:
            keep = np.ones(4, dtype=bool)
        img = img * keep[None, None, :]
        return (torch.from_numpy(img.transpose(2, 0, 1)).float(),
                torch.from_numpy(lab).long(), idh)

rng = np.random.RandomState(CONFIG['seed'])
rng.shuffle(ok)
n_val = max(1, int(len(ok) * CONFIG['val_frac']))
val_subjects, train_subjects = ok[:n_val], ok[n_val:]
train_files = [os.path.join(CONFIG['work_dir'], 'slices', f'{s}.npz') for s in train_subjects]
val_files = [os.path.join(CONFIG['work_dir'], 'slices', f'{s}.npz') for s in val_subjects]
train_ds = SliceDataset(train_files, train=True)
val_ds = SliceDataset(val_files, train=False)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                                       num_workers=4, pin_memory=True, drop_last=True)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False,
                                     num_workers=4, pin_memory=True)
train_idh = np.mean([int(np.load(f)['idh']) for f in train_files])
print(f'train subjects {len(train_files)} slices {len(train_ds)} | val subjects {len(val_files)} slices {len(val_ds)}')
print(f'train IDH-mutant rate {train_idh:.2f}')

## Model

2D U-Net with two heads (simplified from MTamba's multi-task design):
- **Segmentation head:** 4-class output (background, NET, edema, ET)
- **IDH head:** global average pooling + 2-class classifier

Grade head was removed because our 50-subject subset contained only HGG cases.

Model size: ~2.4 M parameters.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.b(x)

class UNet2D(nn.Module):
    def __init__(self):
        super().__init__()
        c = [32, 64, 128, 256]
        self.e1 = ConvBlock(4, c[0])
        self.e2 = ConvBlock(c[0], c[1])
        self.e3 = ConvBlock(c[1], c[2])
        self.e4 = ConvBlock(c[2], c[3])
        self.p = nn.MaxPool2d(2)
        self.b = ConvBlock(c[3], c[3])
        self.u3 = nn.ConvTranspose2d(c[3], c[2], 2, stride=2)
        self.d3 = ConvBlock(c[3], c[2])
        self.u2 = nn.ConvTranspose2d(c[2], c[1], 2, stride=2)
        self.d2 = ConvBlock(c[2], c[1])
        self.u1 = nn.ConvTranspose2d(c[1], c[0], 2, stride=2)
        self.d1 = ConvBlock(c[1], c[0])
        self.seg = nn.Conv2d(c[0], 4, 1)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.idh = nn.Sequential(nn.Dropout(0.3), nn.Linear(c[3], 2))

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.p(e1))
        e3 = self.e3(self.p(e2))
        e4 = self.e4(self.p(e3))
        b = self.b(self.p(e4))
        d3 = self.d3(torch.cat([self.u3(b), e4], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e3], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e2], 1))
        return self.seg(d1), self.idh(self.gap(b).view(x.size(0), -1))

model = UNet2D().to(device)
sum(p.numel() for p in model.parameters()) / 1e6

## Training

Joint loss: weighted cross-entropy for segmentation + weighted cross-entropy for IDH.

IDH class weight compensates for class imbalance (mutant rate ~7 % in training set).

In [ ]:
seg_w = torch.tensor([0.1, 1.0, 1.0, 1.0], device=device)
idh_w = torch.tensor([1.0, 3.97], device=device)
ce_seg = nn.CrossEntropyLoss(weight=seg_w, ignore_index=0)
ce_idh = nn.CrossEntropyLoss(weight=idh_w)

opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])

best = 0.0
ckpt = os.path.join(CONFIG['work_dir'], 'best.pt')

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    tl, ti = 0.0, 0.0
    for img, lab, idh in train_dl:
        img, lab, idh = img.to(device), lab.to(device), torch.tensor(idh, device=device)
        opt.zero_grad()
        seg_out, idh_out = model(img)
        ls = ce_seg(seg_out, lab) + ce_idh(idh_out, idh)
        ls.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tl += ls.item()
        ti += ce_idh(idh_out, idh).item()
    sched.step()

    model.eval()
    vd, vi = 0.0, 0.0
    n = 0
    with torch.no_grad():
        for img, lab, idh in val_dl:
            img, lab, idh = img.to(device), lab.to(device), torch.tensor(idh, device=device)
            seg_out, idh_out = model(img)
            vd += ce_seg(seg_out, lab).item()
            vi += ce_idh(idh_out, idh).item()
            n += 1
    vd /= max(n, 1)
    vi /= max(n, 1)
    print(f'epoch {epoch:02d} | train seg {tl/len(train_dl):.3f} idh {ti/len(train_dl):.3f} | val seg {vd:.3f} idh {vi:.3f}')

    cur = 1.0 - vd
    if cur > best:
        best = cur
        torch.save({'model': model.state_dict(), 'epoch': epoch, 'best': best}, ckpt)
        print('  -> saved')

## Evaluation

### Missing-modality robustness

Test 4 configurations (from MTamba paper's robustness analysis):
- `full-4`: all sequences available
- `flair-only`: only FLAIR (simulates contrast allergy)
- `t2+flair`: no contrast, no T1
- `no-t1ce`: everything except gadolinium-enhanced T1

In [ ]:
def slice_dice(pred, tgt, smooth=1e-5):
    pred = pred.argmax(1)
    dice = {}
    for k, (a, b) in enumerate([(1, 2), (1, 3), (3, 4)], 1):
        pk = ((pred == a) | (pred == b)).float()
        tk = ((tgt == a) | (tgt == b)).float()
        dice[['WT', 'TC', 'ET'][k - 1]] = float((2 * (pk * tk).sum() / (pk.sum() + tk.sum() + smooth)).cpu())
    return dice

MODES = {
    'full-4': [1, 1, 1, 1],
    'flair-only': [1, 0, 0, 0],
    't2+flair': [1, 0, 0, 1],
    'no-t1ce': [1, 0, 1, 1]
}

ck = torch.load(ckpt, map_location=device)
model.load_state_dict(ck['model'])
model.eval()

rows = []
subj_probs = {m: {} for m in MODES}

with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
    for mode, keep in MODES.items():
        dices = {'WT': [], 'TC': [], 'ET': []}
        for f in val_files:
            with np.load(f) as z:
                img, lab = z['image'], z['label']
            x = torch.from_numpy(img.transpose(0, 3, 1, 2)).float().to(device)
            x = x * torch.tensor(keep, device=device)[None, :, None, None]
            probs_idh = []
            for i in range(0, len(x), 8):
                seg, p_idh = model(x[i:i + 8])
                d = slice_dice(seg.float(), torch.from_numpy(lab[i:i + 8]).to(device))
                for k in dices:
                    dices[k].append(d[k])
                probs_idh.append(F.softmax(p_idh.float(), 1)[:, 1].cpu().numpy())
            subj_probs[mode][os.path.basename(f)] = float(np.concatenate(probs_idh).mean())
        row = {'mode': mode}
        for k in dices:
            row[k] = float(np.mean(dices[k]))
        rows.append(row)

seg_df = pd.DataFrame(rows).set_index('mode').loc[list(MODES)]
seg_df.round(4)

### IDH classification results

Bootstrap 1000 iterations for 90 % confidence intervals on AUC and accuracy.

In [ ]:
def bootstrap(y_true, score, pred, n=1000):
    y_true, score, pred = map(np.asarray, (y_true, score, pred))
    auc = roc_auc_score(y_true, score)
    acc = accuracy_score(y_true, pred)
    rng = np.random.RandomState(42)
    aucs, accs = [], []
    for _ in range(n):
        idx = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], score[idx]))
        accs.append(accuracy_score(y_true[idx], pred[idx]))
    aucs, accs = np.sort(aucs), np.sort(accs)
    return (auc, aucs[int(0.05 * len(aucs))], aucs[int(0.95 * len(aucs))]), \n           (acc, accs[int(0.05 * len(accs))], accs[int(0.95 * len(accs))])

idh_true = {os.path.basename(f): int(np.load(f)['idh']) for f in val_files}
diag_rows = []

for mode in MODES:
    sids = list(subj_probs[mode])
    yt = np.array([idh_true[s] for s in sids])
    sc = np.array([subj_probs[mode][s] for s in sids])
    pr = (sc >= 0.5).astype(int)
    auc, acc = bootstrap(yt, sc, pr)
    diag_rows.append({
        'mode': mode,
        'IDH_AUC': round(auc[0], 3),
        'AUC_CI': f'[{auc[1]:.3f},{auc[2]:.3f}]',
        'IDH_ACC': round(acc[0], 3),
        'n_val': len(sids)
    })

diag_df = pd.DataFrame(diag_rows).set_index('mode')
diag_df

### Uncertainty triage curve

MC-dropout (Gal & Ghahramani, 2016): run 20 stochastic forward passes with dropout enabled at inference.

Confidence = max class probability averaged over passes.
Subjects below a confidence threshold are flagged for radiologist review.

Goal: high accuracy on auto-accepted cases while minimizing unnecessary referrals.

In [ ]:
def enable_mc_dropout(m):
    for mod in m.modules():
        if isinstance(mod, nn.Dropout):
            mod.train()

model.eval()
enable_mc_dropout(model)

keep = torch.tensor(MODES['full-4'], device=device)
T = CONFIG['mc_passes']
all_probs = {os.path.basename(f): [] for f in val_files}

with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
    for t in range(T):
        for f in val_files:
            with np.load(f) as z:
                img = z['image']
            x = torch.from_numpy(img.transpose(0, 3, 1, 2)).float().to(device)
            x = x * keep[None, :, None, None]
            ps = []
            for i in range(0, len(x), 8):
                _, p_idh = model(x[i:i + 8])
                ps.append(F.softmax(p_idh.float(), 1).cpu().numpy())
            all_probs[os.path.basename(f)].append(np.concatenate(ps).mean(0))

probs = {s: np.mean(v, 0) for s, v in all_probs.items()}
sids = list(probs)
yt = np.array([idh_true[s] for s in sids])
conf = np.array([probs[s].max() for s in sids])
pred = np.array([probs[s].argmax() for s in sids])

ths = np.linspace(0.5, 0.95, 19)
cov, acc = [], []

for th in ths:
    m = conf >= th
    cov.append(m.mean())
    acc.append((pred[m] == yt[m]).mean() if m.any() else np.nan)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cov, acc, 'o-')
for x_, y_, t_ in zip(cov, acc, ths):
    ax.annotate(f'{t_:.2f}', (x_, y_), fontsize=7, alpha=0.7)
ax.set_xlabel('coverage (fraction of subjects auto-accepted)')
ax.set_ylabel('IDH accuracy on accepted')
ax.set_title('Uncertainty triage: reject low-confidence -> refer to radiologist')
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(CONFIG['work_dir'], 'triage_curve.png'), dpi=200)
print('baseline acc all-accept:', (pred == yt).mean())
list(zip(ths.round(2), np.round(cov, 2), np.round(acc, 2)))[:8]

## Save results

In [ ]:
seg_df.to_csv(os.path.join(CONFIG['work_dir'], 'seg_results.csv'))
diag_df.to_csv(os.path.join(CONFIG['work_dir'], 'idh_results.csv'))

summary = {
    'train_subjects': len(train_files),
    'val_subjects': len(val_files),
    'best_val_wt_dice': float(best),
    'epochs': CONFIG['epochs']
}

with open(os.path.join(CONFIG['work_dir'], 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

seg_df.round(4), diag_df